# XBP1-es Detection Across TCGA Cancer Types

**Title:** A topology-dependent IRE1α signaling axis generates a cryptic XBP1 isoform present in human cancers  
**Authors:** Jose Roberto Rodrigues Reyes¹, Collin Vergel De Dios², Debalina Datta³, Taivan Batjargal⁴, Surenna Pecchia², Feroz R. Papa³, Maxwell Z. Wilson^{1,2}*  
**Affiliations:**  
¹ Interdisciplinary Program in Quantitative Biosciences, University of California, Santa Barbara, CA, USA  
² Department of Molecular, Cellular, and Developmental Biology, University of California Santa Barbara, CA, USA  
³ Neuroscience Research Institute, University of California Santa Barbara, CA, USA  
⁴ Department of Medicine, University of California, San Francisco, CA, USA  
*Corresponding author: Maxwell Z. Wilson. mzw@ucsb.edu  
**Repository:** Supplementary Code

---

## Background

### XBP1 isoforms — important distinctions

The XBP1 gene produces three functionally distinct isoforms that must not be confused:

**XBP1-u (unspliced):** The baseline, unprocessed transcript. Encodes a short, unstable protein that lacks transcriptional activation activity.

**XBP1-s (spliced):** Produced by the unconventional, IRE1α-catalyzed cytoplasmic splicing of XBP1-u mRNA. IRE1α's endonuclease domain excises a precise **26-nucleotide intron within exon 4**, causing a frameshift that produces a longer, stable, and potent transcriptional activator of UPR target genes. This is the canonical IRE1α output and the primary effector of the adaptive UPR.

**XBP1-es (exon-skipped):** A distinct isoform in which **the entire exon 4 (146 nt) is removed** by the cellular spliceosome through conventional alternative splicing — not by IRE1α's endonuclease activity. This exon-skipping event produces a transcript that retains broad UPR transcription factor activity but, critically, **fails to induce integrated stress response (ISR) genes** including ATF4 and DDIT3/CHOP. XBP1-es therefore functionally decouples IRE1α signaling from the PERK/ATF4/CHOP apoptotic axis.

> **Key distinction:** XBP1-s results from IRE1α cutting out 26 nt *within* exon 4. XBP1-es results from the spliceosome skipping exon 4 *entirely*. These are mechanistically and functionally different events, detectable by different splice junctions in RNA-seq data.

### Biological context

XBP1-es was first described in chimpanzee cells during rotavirus infection (Duarte et al., *J. Virol.* 2019), where IRE1α inhibition did not prevent its formation, confirming spliceosome rather than IRE1α origin. This work demonstrates that cytoplasmic IRE1α hyperclusters — formed when IRE1α is displaced from the ER membrane — upregulate spliceosome assembly genes and create a splicing environment that promotes XBP1-es formation. XBP1-es is detectable in 10.9% of TCGA tumors, with highest prevalence in acute myeloid leukemia (LAML, 60%) and breast cancer (BRCA, 27%).

This notebook quantifies the XBP1-es exon-skipping junction (exon 3 → exon 5, skipping exon 4 entirely) across all TCGA cancer types using **Snaptron** — a web service providing rapid, sample-level access to splice junction read counts from large RNA-seq compendiums — and the **recount3**-based TCGA v2 compilation (hg38).

### Key references
- This paper: Rodrigues Reyes JR et al. *A topology-dependent IRE1α signaling axis generates a cryptic XBP1 isoform present in human cancers.*
- Snaptron: Wilks C et al. *Bioinformatics* 2018. https://doi.org/10.1093/bioinformatics/bty101  
- recount3: Wilks C et al. *Genome Biology* 2021. https://doi.org/10.1186/s13059-021-02533-6  
- XBP1-es first description: Duarte M et al. *J. Virol.* 2019. https://doi.org/10.1128/jvi.01739-18  
- IRE1α/XBP1-s splicing: Yoshida H et al. *Cell* 2001. https://doi.org/10.1016/S0092-8674(01)00271-9


## 1. Imports and Configuration

All analysis parameters are defined in the `CONFIG` dictionary below. Modify these values to adapt the analysis to a different gene or junction of interest.


In [1]:
import subprocess
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm

# Rebuild font cache to pick up system fonts (run once per environment)
fm.fontManager.__init__()

# ─── Configuration ────────────────────────────────────────────────────────────
CONFIG = {
    # Snaptron compilation (recount3-based TCGA, hg38)
    "compilation": "tcgav2",

    # Chromosome (GRCh38/hg38)
    "chrom": "chr22",

    # XBP1 splice junction coordinates (GRCh38 intron boundaries, 0-based)
    # These represent the last base of the donor exon to the first base of
    # the acceptor exon for each junction.
    "junctions": {
        # Exon-skipping junction: exon 3 → exon 5 (XBP1-es isoform)
        "es":  (28_795_733, 28_797_076),
        # Canonical junction: exon 3 → exon 4
        "e34": (28_796_193, 28_797_076),
        # Canonical junction: exon 4 → exon 5
        "e45": (28_795_733, 28_796_046),
    },

    # Minimum read denominator for PSI calculation
    # Samples with fewer total junction-spanning reads are excluded
    "min_denom": 5,

    # Minimum number of qualifying tumor samples required to include a
    # cancer type in summary figures
    "min_samples_per_cancer": 30,

    # TCGA sample_type values classified as tumor
    # Includes haematological malignancies (LAML) which use peripheral blood
    "tumor_sample_types": {
        "Primary Tumor",
        "Primary Blood Derived Cancer - Peripheral Blood",
        "Metastatic",
        "Recurrent Tumor",
        "Additional - New Primary",
        "Additional Metastatic",
    },

    # Figure formatting (Nature Chemical Biology: 180 mm two-column width)
    "fig_width_mm":  180,
    "fig_height_mm": 140,
    "font_family":   "Liberation Sans",  # Arial-metric-compatible (Linux)
                                          # Replace with "Arial" on macOS/Windows
    "font_size_axis_label": 6,
    "font_size_tick":       5,
    "font_size_panel_label": 8,
    "pdf_fonttype": 42,  # TrueType embedding — text remains editable in Illustrator
}

print("Configuration loaded.")
print(f"  Compilation : {CONFIG['compilation']}")
print(f"  Chromosome  : {CONFIG['chrom']}")
for name, (s, e) in CONFIG['junctions'].items():
    print(f"  Junction {name:>3} : {CONFIG['chrom']}:{s:,}–{e:,}")


Fontconfig warning: ignoring C.UTF-8: not a valid language tag
Matplotlib is building the font cache; this may take a moment.


Configuration loaded.
  Compilation : tcgav2
  Chromosome  : chr22
  Junction  es : chr22:28,795,733–28,797,076
  Junction e34 : chr22:28,796,193–28,797,076
  Junction e45 : chr22:28,795,733–28,796,046


## 2. Junction Read Count Retrieval from Snaptron

Snaptron provides a REST API for querying splice junction read counts across large RNA-seq compendiums. We use the **exact-match** mode (`exact=1`), which returns only samples containing a junction with precisely the specified donor and acceptor coordinates — avoiding spurious partial overlaps.

### Junctions queried

Three junctions are required to calculate XBP1-es PSI:

| Junction | Coordinates (GRCh38) | Biological meaning |
|----------|---------------------|--------------------|
| **XBP1-es** | chr22:28,795,733–28,797,076 | Exon 3 → Exon 5 direct junction (entire exon 4 skipped by spliceosome) |
| **Canonical E3→E4** | chr22:28,796,193–28,797,076 | Exon 3 → Exon 4 (normal splicing) |
| **Canonical E4→E5** | chr22:28,795,733–28,796,046 | Exon 4 → Exon 5 (normal splicing) |

> **Note:** The XBP1-es junction (E3→E5) spans the entire exon 4 skip. It is distinct from the XBP1-s isoform, which is produced by IRE1α removing a 26-nt intron *within* exon 4 — a much smaller event that does not produce an exon 3–exon 5 junction in standard RNA-seq alignment.

For each junction, the API returns a tab-separated response where the 13th field (index 12) encodes per-sample read counts as comma-separated `rail_id:count` pairs. The `rail_id` is Snaptron's internal sample identifier, used to retrieve metadata.

**API endpoint format:**
```
https://snaptron.cs.jhu.edu/{compilation}/snaptron?regions={chrom}:{start}-{end}&exact=1
```


In [2]:
def snaptron_exact(compilation, chrom, start, end):
    """
    Query the Snaptron API for a single splice junction using exact-match mode.

    Parameters
    ----------
    compilation : str
        Snaptron compilation identifier (e.g. 'tcgav2', 'gtexv2').
    chrom : str
        Chromosome in UCSC format (e.g. 'chr22').
    start : int
        Intron start coordinate (GRCh38, 0-based).
    end : int
        Intron end coordinate (GRCh38, 0-based).

    Returns
    -------
    dict
        Mapping of {rail_id (int): read_count (int)} for all samples
        in which this exact junction was detected.
    """
    url = (
        f"https://snaptron.cs.jhu.edu/{compilation}/snaptron"
        f"?regions={chrom}:{start}-{end}&exact=1"
    )
    result = subprocess.run(
        ["curl", "-s", "-L", "--max-time", "120", url],
        capture_output=True, text=True
    )

    if not result.stdout.strip():
        return {}

    lines = result.stdout.strip().split("\n")
    if len(lines) < 2:
        return {}

    # Field index 12 contains the sample:count pairs
    samples_field = lines[1].split("\t")[12]
    counts = {}
    for item in samples_field.strip(",").split(","):
        if ":" in item:
            rail_id, count = item.split(":", 1)
            try:
                counts[int(rail_id)] = int(count)
            except ValueError:
                continue
    return counts


# ── Query all three junctions ─────────────────────────────────────────────────
print("Querying Snaptron TCGA v2 compilation...")
junction_counts = {}
for name, (start, end) in CONFIG["junctions"].items():
    junction_counts[name] = snaptron_exact(
        CONFIG["compilation"], CONFIG["chrom"], start, end
    )
    n = len(junction_counts[name])
    print(f"  Junction {name:>3} ({CONFIG['chrom']}:{start:,}–{end:,}): "
          f"{n:,} samples with ≥1 read")


Querying Snaptron TCGA v2 compilation...
  Junction  es (chr22:28,795,733–28,797,076): 1,627 samples with ≥1 read
  Junction e34 (chr22:28,796,193–28,797,076): 11,337 samples with ≥1 read
  Junction e45 (chr22:28,795,733–28,796,046): 11,343 samples with ≥1 read


## 3. Sample Metadata Retrieval

Sample metadata is retrieved from the Snaptron `/samples` endpoint in batches of 500 IDs to avoid URL length limits. For TCGA v2, the relevant metadata fields are extracted by fixed column index from the tab-separated response:

| Field index | Content |
|-------------|---------|
| 1 | `rail_id` — Snaptron internal sample ID |
| 3 | `cancer_abbrev` — TCGA cancer type abbreviation (e.g. BRCA, LUAD) |
| 6 | `cancer_full` — Full cancer type name |
| 11 | `sample_type` — Tissue type (e.g. Primary Tumor, Solid Tissue Normal) |

> **Note on LAML:** Acute Myeloid Leukemia samples have `sample_type = "Primary Blood Derived Cancer - Peripheral Blood"` rather than `"Primary Tumor"`. This is accounted for in the tumor sample type whitelist in the configuration above.


In [3]:
def fetch_metadata(compilation, sample_ids, chunk_size=500):
    """
    Retrieve sample metadata from the Snaptron /samples endpoint.

    Parameters
    ----------
    compilation : str
        Snaptron compilation identifier.
    sample_ids : iterable of int
        rail_id values to retrieve metadata for.
    chunk_size : int
        Number of IDs per API request (default 500).

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by rail_id with columns:
        cancer_abbrev, cancer_full, sample_type.
    """
    records = []
    ids = list(sample_ids)

    for i in range(0, len(ids), chunk_size):
        batch = ids[i : i + chunk_size]
        ids_str = ",".join(str(x) for x in batch)
        url = (
            f"https://snaptron.cs.jhu.edu/{compilation}/samples"
            f"?ids={ids_str}"
        )
        result = subprocess.run(
            ["curl", "-s", "-L", "--max-time", "60", url],
            capture_output=True, text=True
        )
        # Skip header line; parse fixed field indices
        for line in result.stdout.strip().split("\n")[1:]:
            fields = line.split("\t")
            if len(fields) < 12:
                continue
            records.append({
                "rail_id":       fields[1],
                "cancer_abbrev": fields[3],
                "cancer_full":   fields[6],
                "sample_type":   fields[11],
            })

    df = pd.DataFrame(records)
    # Keep only rows with valid numeric rail_ids
    df = df[df["rail_id"].str.isdigit()].copy()
    df["rail_id"] = df["rail_id"].astype(int)
    return df.set_index("rail_id")


# ── Collect all unique sample IDs across the three junctions ─────────────────
all_sample_ids = set()
for counts in junction_counts.values():
    all_sample_ids.update(counts.keys())

print(f"Fetching metadata for {len(all_sample_ids):,} unique samples...")
metadata = fetch_metadata(CONFIG["compilation"], all_sample_ids)
print(f"  Metadata retrieved: {len(metadata):,} samples")
print(f"  Cancer types found: {metadata['cancer_abbrev'].nunique()}")
print(f"\nSample type breakdown:")
print(metadata["sample_type"].value_counts().to_string())


Fetching metadata for 11,347 unique samples...
  Metadata retrieved: 11,347 samples
  Cancer types found: 33

Sample type breakdown:
sample_type
Primary Tumor                                      9960
Solid Tissue Normal                                 740
Metastatic                                          393
Primary Blood Derived Cancer - Peripheral Blood     126
NA                                                   65
Recurrent Tumor                                      49
Additional - New Primary                             11
                                                      2
Additional Metastatic                                 1


## 4. PSI Calculation

**Percent Spliced In (PSI)** quantifies the fraction of XBP1 transcripts at this locus that use the exon-skipping isoform (XBP1-es). We use the standard cassette-exon PSI formula, consistent with rMATS convention:

$$\text{PSI} = \frac{C_{\text{skip}}}{C_{\text{skip}} + \dfrac{C_{\text{E3→E4}} + C_{\text{E4→E5}}}{2}}$$

where:
- $C_{\text{skip}}$ = reads spanning the exon 3→exon 5 junction (XBP1-es; entire exon 4 skipped)
- $C_{\text{E3→E4}}$ = reads spanning the canonical exon 3–exon 4 junction
- $C_{\text{E4→E5}}$ = reads spanning the canonical exon 4–exon 5 junction

The denominator averages the two flanking canonical junctions to account for unequal read coverage across junction boundaries. Samples with a denominator < `min_denom` (default: 5) are excluded to avoid unreliable PSI estimates from very low-coverage loci.

PSI values range from 0 (all reads support canonical splicing) to 1 (all reads support exon skipping). Values are reported as percentages (PSI × 100) in figures.

> **Interpretation note:** Because XBP1-es is a rare, spliceosome-mediated alternative splicing event in bulk tissue (where most cells are not actively producing XBP1-es), mean PSI values across cancer types are typically < 1%. This does not reflect low biological significance — it reflects dilution across a heterogeneous cell population. In cells actively producing XBP1-es, local PSI would be substantially higher.


In [4]:
def calculate_psi(junction_counts, metadata, min_denom=5):
    """
    Calculate per-sample XBP1-es PSI and join with sample metadata.

    Parameters
    ----------
    junction_counts : dict of {str: dict}
        Keys: 'es', 'e34', 'e45'. Values: {rail_id: read_count}.
    metadata : pd.DataFrame
        Sample metadata indexed by rail_id.
    min_denom : int
        Minimum denominator (skip + mean canonical) to include a sample.

    Returns
    -------
    pd.DataFrame
        Per-sample DataFrame with columns:
        es_reads, e34_reads, e45_reads, psi,
        cancer_abbrev, cancer_full, sample_type.
    """
    # Union of all sample IDs across junctions
    all_ids = set().union(*[set(v) for v in junction_counts.values()])

    records = []
    for sid in all_ids:
        c_skip = junction_counts["es"].get(sid, 0)
        c_e34  = junction_counts["e34"].get(sid, 0)
        c_e45  = junction_counts["e45"].get(sid, 0)

        denominator = c_skip + (c_e34 + c_e45) / 2.0
        if denominator < min_denom:
            continue

        records.append({
            "rail_id":   sid,
            "es_reads":  c_skip,
            "e34_reads": c_e34,
            "e45_reads": c_e45,
            "psi":       c_skip / denominator,
        })

    psi_df = pd.DataFrame(records).set_index("rail_id")
    return psi_df.join(metadata, how="left")


def aggregate_psi(df, group_col):
    """
    Compute per-group PSI summary statistics.

    Parameters
    ----------
    df : pd.DataFrame
        Per-sample PSI DataFrame (output of calculate_psi).
    group_col : str or list
        Column(s) to group by (e.g. 'cancer_abbrev').

    Returns
    -------
    pd.DataFrame
        Per-group summary with columns:
        n_samples, n_with_es, mean_psi, median_psi, pct_detected.
    """
    return (
        df.groupby(group_col)
        .agg(
            n_samples    = ("psi",      "count"),
            n_with_es    = ("es_reads", lambda x: (x > 0).sum()),
            mean_psi     = ("psi",      "mean"),
            median_psi   = ("psi",      "median"),
            pct_detected = ("es_reads", lambda x: 100 * (x > 0).mean()),
        )
        .sort_values("mean_psi", ascending=False)
        .reset_index()
    )


# ── Calculate PSI for all samples ────────────────────────────────────────────
psi_all = calculate_psi(junction_counts, metadata, min_denom=CONFIG["min_denom"])
print(f"Samples passing PSI filter (denominator ≥ {CONFIG['min_denom']}): "
      f"{len(psi_all):,}")

# ── Stratify by sample type ───────────────────────────────────────────────────
psi_tumor  = psi_all[psi_all["sample_type"].isin(CONFIG["tumor_sample_types"])].copy()
psi_normal = psi_all[psi_all["sample_type"] == "Solid Tissue Normal"].copy()

print(f"  Tumor samples  : {len(psi_tumor):,}")
print(f"  Normal samples : {len(psi_normal):,}")

# ── Aggregate by cancer type ──────────────────────────────────────────────────
agg_tumor  = aggregate_psi(psi_tumor,  "cancer_abbrev")
agg_normal = aggregate_psi(psi_normal, "cancer_abbrev")

# Apply minimum sample threshold
agg_tumor_filtered = agg_tumor[
    agg_tumor["n_samples"] >= CONFIG["min_samples_per_cancer"]
].copy()

print(f"\nCancer types with ≥{CONFIG['min_samples_per_cancer']} tumor samples: "
      f"{len(agg_tumor_filtered)}")
print("\nTop 10 cancer types by mean XBP1-es PSI (%):")
top10 = agg_tumor_filtered.head(10).copy()
top10["mean_psi_pct"] = top10["mean_psi"] * 100
print(top10[["cancer_abbrev", "n_samples", "mean_psi_pct", "pct_detected"]]
      .to_string(index=False))


Samples passing PSI filter (denominator ≥ 5): 11,324
  Tumor samples  : 10,517
  Normal samples : 740

Cancer types with ≥30 tumor samples: 33

Top 10 cancer types by mean XBP1-es PSI (%):
cancer_abbrev  n_samples  mean_psi_pct  pct_detected
         LAML        126      0.541050     63.492063
         BLCA        414      0.125726      9.661836
         TGCT        156      0.124600     14.102564
         HNSC        503      0.099790     16.898608
         CESC        305      0.081764     13.442623
         THYM        117      0.077732      9.401709
         DLBC         48      0.071287      8.333333
         MESO         87      0.061135     11.494253
          LGG        531      0.059647      4.331450
         STAD        415      0.051140     21.927711


## 5. Save Output Tables

Three CSV files are written:

| File | Contents |
|------|----------|
| `tcga_XBP1es_PSI_per_sample.csv` | Per-sample PSI values with cancer type and sample type annotations |
| `tcga_XBP1es_PSI_by_cancer_tumor.csv` | Per-cancer-type summary statistics for tumor samples |
| `tcga_XBP1es_PSI_by_cancer_normal.csv` | Per-cancer-type summary statistics for solid normal samples |


In [5]:
# Per-sample table (reset index to include rail_id as a column)
psi_all.reset_index().to_csv("tcga_XBP1es_PSI_per_sample.csv", index=False)

# Per-cancer-type summaries
agg_tumor.to_csv("tcga_XBP1es_PSI_by_cancer_tumor.csv", index=False)
agg_normal.to_csv("tcga_XBP1es_PSI_by_cancer_normal.csv", index=False)

print("Saved:")
print("  tcga_XBP1es_PSI_per_sample.csv")
print("  tcga_XBP1es_PSI_by_cancer_tumor.csv")
print("  tcga_XBP1es_PSI_by_cancer_normal.csv")


Saved:
  tcga_XBP1es_PSI_per_sample.csv
  tcga_XBP1es_PSI_by_cancer_tumor.csv
  tcga_XBP1es_PSI_by_cancer_normal.csv


## 6. Figure Generation

Two panels are produced in a single figure formatted for *Nature Chemical Biology* (180 mm wide, 5–7 pt fonts, TrueType font embedding for Illustrator editability):

- **Panel a** — Ranked horizontal bar chart of mean XBP1-es PSI (%) across TCGA cancer types (tumor samples only). XBP1-es is the exon 4-skipped isoform detected by the exon 3→exon 5 junction. Diamond markers on the secondary x-axis show the percentage of samples in which at least one XBP1-es read was detected.

- **Panel b** — Heatmap showing the percentage of samples with XBP1-es detected, comparing TCGA tumor vs. TCGA solid normal side by side for each cancer type. Grey cells indicate cancer types without solid normal samples (e.g. LAML).

> **Font note:** Liberation Sans is used here (Arial-metric-compatible, available on Linux). On macOS or Windows, replace `"Liberation Sans"` with `"Arial"` in the CONFIG dictionary.


In [6]:
# ── Apply matplotlib style ────────────────────────────────────────────────────
MM = 1 / 25.4  # mm to inches conversion

plt.rcParams.update({
    "font.family":        CONFIG["font_family"],
    "font.size":          CONFIG["font_size_axis_label"],
    "axes.labelsize":     CONFIG["font_size_axis_label"],
    "axes.titlesize":     CONFIG["font_size_axis_label"],
    "xtick.labelsize":    CONFIG["font_size_tick"],
    "ytick.labelsize":    CONFIG["font_size_tick"],
    "legend.fontsize":    CONFIG["font_size_tick"],
    "axes.linewidth":     0.5,
    "xtick.major.width":  0.5,
    "ytick.major.width":  0.5,
    "xtick.major.size":   2,
    "ytick.major.size":   2,
    "lines.linewidth":    0.75,
    "patch.linewidth":    0.5,
    "pdf.fonttype":       CONFIG["pdf_fonttype"],
    "ps.fonttype":        CONFIG["pdf_fonttype"],
})

C_TUMOR  = "#0279EE"   # Blue — TCGA tumor
C_NORMAL = "#FF9400"   # Orange — TCGA solid normal
C_DARK   = "#222222"   # Dark grey — secondary markers

# ── Prepare data ──────────────────────────────────────────────────────────────
# Panel a: ranked tumor bars (already filtered to ≥ min_samples_per_cancer)
bar_data = agg_tumor_filtered.sort_values("mean_psi", ascending=True).copy()
bar_data["psi_pct"] = bar_data["mean_psi"] * 100

# Panel b: heatmap — merge tumor and normal pct_detected
hm_tumor  = agg_tumor_filtered.set_index("cancer_abbrev")[["pct_detected"]].rename(
    columns={"pct_detected": "TCGA tumor"}
)
hm_normal = agg_normal.set_index("cancer_abbrev")[["pct_detected"]].rename(
    columns={"pct_detected": "TCGA solid normal"}
)
hm_data = hm_tumor.join(hm_normal, how="left")
# Sort by tumor pct_detected descending (same order as bar chart, reversed)
hm_data = hm_data.loc[bar_data["cancer_abbrev"].values[::-1]]

# ── Build figure ──────────────────────────────────────────────────────────────
fig_w = CONFIG["fig_width_mm"]  * MM
fig_h = CONFIG["fig_height_mm"] * MM

fig = plt.figure(figsize=(fig_w, fig_h))
gs = gridspec.GridSpec(
    1, 2, figure=fig,
    left=0.08, right=0.97,
    top=0.95,  bottom=0.06,
    wspace=0.50,
    width_ratios=[1.4, 0.55],
)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])

# ── Panel a: ranked bar chart ─────────────────────────────────────────────────
y = np.arange(len(bar_data))

ax_a.barh(y, bar_data["psi_pct"],
          color=C_TUMOR, alpha=0.85, height=0.65, zorder=3, linewidth=0)
ax_a.set_yticks(y)
ax_a.set_yticklabels(bar_data["cancer_abbrev"], fontsize=CONFIG["font_size_tick"])
ax_a.set_xlabel("Mean XBP1-es PSI (%)", fontsize=CONFIG["font_size_axis_label"],
                color=C_TUMOR)
ax_a.tick_params(axis="x", colors=C_TUMOR, labelsize=CONFIG["font_size_tick"])
ax_a.set_xlim(0, bar_data["psi_pct"].max() * 1.20)
ax_a.grid(axis="x", linestyle=":", linewidth=0.3, alpha=0.6, zorder=0)
ax_a.spines[["top", "right"]].set_visible(False)
ax_a.spines["bottom"].set_color(C_TUMOR)

# Secondary x-axis: % samples detected (diamond markers)
ax_a2 = ax_a.twiny()
ax_a2.scatter(bar_data["pct_detected"], y,
              color=C_DARK, s=5, zorder=5, marker="D", alpha=0.75, linewidths=0)
ax_a2.set_xlabel("% samples detected",
                 fontsize=CONFIG["font_size_tick"], color=C_DARK, labelpad=2)
ax_a2.tick_params(axis="x", labelsize=CONFIG["font_size_tick"],
                  colors=C_DARK, pad=1)
ax_a2.set_xlim(0, 75)
ax_a2.spines["top"].set_linewidth(0.5)

ax_a.text(-0.14, 1.04, "a", transform=ax_a.transAxes,
          fontsize=CONFIG["font_size_panel_label"], fontweight="bold", va="top")

# ── Panel b: heatmap ──────────────────────────────────────────────────────────
data_matrix = hm_data.values.astype(float)
masked      = np.ma.masked_invalid(data_matrix)

cmap = plt.cm.YlOrRd.copy()
cmap.set_bad("#E8E8E8")
im = ax_b.imshow(masked, aspect="auto", cmap=cmap, vmin=0, vmax=65)

ax_b.set_xticks([0, 1])
ax_b.set_xticklabels(["TCGA\ntumor", "TCGA\nsolid normal"],
                     fontsize=CONFIG["font_size_tick"])
ax_b.set_yticks(np.arange(len(hm_data)))
ax_b.set_yticklabels(hm_data.index, fontsize=CONFIG["font_size_tick"])
ax_b.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False,
                 pad=1, length=2)

# Annotate each cell with its numeric value
for i in range(data_matrix.shape[0]):
    for j in range(data_matrix.shape[1]):
        val = data_matrix[i, j]
        if not np.isnan(val):
            text_color = "white" if val > 38 else C_DARK
            ax_b.text(j, i, f"{val:.0f}",
                      ha="center", va="center",
                      fontsize=4, color=text_color, fontweight="bold")
        else:
            ax_b.text(j, i, "n/a",
                      ha="center", va="center",
                      fontsize=3.5, color="#999999")

# Grid lines between cells
for i in range(data_matrix.shape[0] + 1):
    ax_b.axhline(i - 0.5, color="white", lw=0.4)
for j in range(data_matrix.shape[1] + 1):
    ax_b.axvline(j - 0.5, color="white", lw=0.4)

# Colorbar
cbar = plt.colorbar(im, ax=ax_b, fraction=0.08, pad=0.04, shrink=0.5)
cbar.set_label("% samples\ndetected", fontsize=CONFIG["font_size_tick"], labelpad=2)
cbar.ax.tick_params(labelsize=4.5, length=2, width=0.4)
cbar.outline.set_linewidth(0.4)

ax_b.text(-0.30, 1.04, "b", transform=ax_b.transAxes,
          fontsize=CONFIG["font_size_panel_label"], fontweight="bold", va="top")

# ── Save ──────────────────────────────────────────────────────────────────────
plt.savefig("XBP1es_TCGA_figure.pdf", bbox_inches="tight", dpi=600, transparent=True)
plt.savefig("XBP1es_TCGA_figure.png", bbox_inches="tight", dpi=300)
plt.close()
print("Figure saved: XBP1es_TCGA_figure.pdf / .png")


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

Figure saved: XBP1es_TCGA_figure.pdf / .png
